In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, to_timestamp, when, element_at, split, regexp_extract
from pathlib import Path
import sqlite3
import pandas as pd

In [2]:
spark = SparkSession.builder \
    .appName("BusETLPipeline") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
print("Spark Session Created with 4 partitions")

Spark Session Created with 4 partitions


In [3]:
base = r"E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV"
print("Loading datasets...")

Loading datasets...


In [4]:
timetable  = spark.read.csv(f"{base}\\clean_timetabledata.csv", header=True, inferSchema=True)
location   = spark.read.csv(f"{base}\\compiled_location_data.csv", header=True, inferSchema=True)
fares      = spark.read.csv(f"{base}\\clean_fares.csv", header=True, inferSchema=True)
disruption = spark.read.csv(f"{base}\\clean_disruptiondata.csv", header=True, inferSchema=True)

In [5]:
print(f"Timetable: {timetable.count()} | Location: {location.count()} | "
      f"Fares: {fares.count()} | Disruption: {disruption.count()}")

Timetable: 176615 | Location: 742 | Fares: 63428 | Disruption: 170


In [6]:
timetable = timetable.repartition(4).cache()

In [7]:
timetable  = timetable.withColumn("line_code", col("LineName"))
fares      = fares.withColumn("line_code", element_at(split(col("LineID"), ":"), -1))
location   = location.withColumn("line_code", col("LineRef"))
disruption = disruption.withColumn("line_code", col("LineRef"))

In [8]:
timetable = timetable.withColumn(
    "RunTimeMinutes",
    regexp_extract(col("RunTime"), r'(\d+)M', 1).cast("double")
)

In [9]:
location = location \
    .withColumn("RecordedAtTime_ts", to_timestamp(col("RecordedAtTime"))) \
    .withColumn("AimedDeparture_ts", to_timestamp(col("OriginAimedDepartureTime"))) \
    .withColumn("Hour", hour(col("RecordedAtTime_ts"))) \
    .withColumn("DelayInMinutes",
        when(col("RecordedAtTime_ts") > col("AimedDeparture_ts"),
             (col("RecordedAtTime_ts").cast("long") - col("AimedDeparture_ts").cast("long")) / 60
        ).otherwise(0)
    )

In [10]:
timetable.createOrReplaceTempView("timetable")
location.createOrReplaceTempView("location")
fares.createOrReplaceTempView("fares")
disruption.createOrReplaceTempView("disruption")

In [11]:
joined_df = spark.sql("""
    WITH tt_agg AS (
        SELECT line_code,
               COUNT(DISTINCT JourneyCode) AS n_scheduled_trips,
               ROUND(AVG(RunTimeMinutes), 2) AS avg_run_time_minutes
        FROM timetable GROUP BY line_code
    ),
    loc_agg AS (
        SELECT line_code,
               COUNT(DISTINCT DatedVehicleJourneyRef) AS n_avl_observations,
               ROUND(AVG(DelayInMinutes), 2) AS avg_delay_minutes
        FROM location GROUP BY line_code
    ),
    disr_agg AS (
        SELECT line_code, COUNT(DISTINCT SituationNumber) AS n_disruptions
        FROM disruption GROUP BY line_code
    ),
    fares_agg AS (
        SELECT line_code, ROUND(AVG(Amount), 2) AS avg_fare
        FROM fares GROUP BY line_code
    )
    SELECT t.line_code,
           t.n_scheduled_trips, t.avg_run_time_minutes,
           COALESCE(l.n_avl_observations, 0) AS n_avl_observations,
           COALESCE(l.avg_delay_minutes, 0.0) AS avg_delay_minutes,
           COALESCE(d.n_disruptions, 0) AS n_disruptions,
           COALESCE(f.avg_fare, 0.0) AS avg_fare
    FROM tt_agg t
    LEFT JOIN loc_agg l ON t.line_code = l.line_code
    LEFT JOIN disr_agg d ON t.line_code = d.line_code
    LEFT JOIN fares_agg f ON t.line_code = f.line_code
    ORDER BY n_disruptions DESC
""")

In [12]:
print("ETL Transformation Completed")
print(f"Final line-level records: {joined_df.count()}")
print(f"Partitions: {joined_df.rdd.getNumPartitions()}")

ETL Transformation Completed
Final line-level records: 42
Partitions: 1


In [13]:
output_dir = Path("E:\school(softwarica)\Sem-4\BigDataProject\Data\Processed")
output_dir.mkdir(parents=True, exist_ok=True)

In [14]:
joined_df.write.mode("overwrite").parquet(str(output_dir / "line_level_summary.parquet"))

In [15]:
joined_pd = joined_df.toPandas()
conn = sqlite3.connect("BODSDatabase.db")
joined_pd.to_sql("etl_line_summary", conn, if_exists="replace", index=False)
conn.commit()
conn.close()


In [16]:
print("Data saved to Parquet and BODSDatabase.db (table: etl_line_summary)")
joined_df.show(5)
joined_df.printSchema()

Data saved to Parquet and BODSDatabase.db (table: etl_line_summary)
+---------+-----------------+--------------------+------------------+-----------------+-------------+--------+
|line_code|n_scheduled_trips|avg_run_time_minutes|n_avl_observations|avg_delay_minutes|n_disruptions|avg_fare|
+---------+-----------------+--------------------+------------------+-----------------+-------------+--------+
|       2A|              355|                 1.2|                15|            32.45|            7|    1.64|
|       B3|              124|                1.04|                 1|            26.42|            7|    1.82|
|      200|               50|                1.89|                 4|            75.96|            7|    4.35|
|       B9|              272|                1.16|                 9|            24.42|            7|    1.86|
|     TUBE|              559|                6.97|                52|           100.54|            6|   14.43|
+---------+-----------------+---------------